# 📊 RAG Retrieval Metrics

**Day 9 — Notebook 3 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- Why generation quality is insufficient — you must also evaluate retrieval
- Precision@K and Recall@K — the two fundamental retrieval metrics
- Mean Reciprocal Rank (MRR) — measuring where the best result appears
- Normalised Discounted Cumulative Gain (NDCG) — graded relevance
- Context Relevance — is what we retrieved actually useful for the LLM?

---

## Why Retrieval Metrics Matter

You can have perfect generation but terrible retrieval:

```
Query: "What is the overtime pay rate?"

Retrieved: Paragraph about standard pay → wrong paragraph, right doc
            Department structure chart → wrong entirely
            Benefits summary → tangentially related

Generated: A confident-sounding hallucination.

Without retrieval metrics, you'd never diagnose this as a retrieval problem.
```

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb

In [ ]:
import sys, os, math
import chromadb
from statistics import mean

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from pydantic import BaseModel

from google.genai import errors

try:
    client = genai.Client(api_key=get_api_key())
except errors.APIError as e:
    print(f"Gemini API Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
else:
    MODEL_ID = "gemini-2.5-flash"
    print("✅ Ready!")
EMBEDDING_MODEL = "gemini-embedding-001"
GEN_MODEL = "gemini-2.5-flash"

---

## 1. Test Corpus and Relevance Labels

To measure retrieval quality, you need to know which documents *should* be retrieved for each query.

In [ ]:
# HR policy corpus — 12 documents
CORPUS = [
    {"id": "annual-leave",      "text": "Employees receive 25 days annual leave per year, accruing at 2.08 days per month. Leave must be requested 5 days in advance via the HR portal."},
    {"id": "sick-leave",        "text": "Employees are entitled to 10 days paid sick leave per year. A doctor's note is required for absences over 3 consecutive days. Sick leave does not carry over."},
    {"id": "parental-leave",    "text": "Primary caregivers receive 18 weeks fully paid parental leave. Secondary caregivers receive 4 weeks. Employees need 6 months tenure to qualify."},
    {"id": "remote-work",       "text": "Employees may work remotely up to 3 days per week. All employees must be in the office on Wednesdays. International remote work is limited to 20 days per year."},
    {"id": "office-equipment",  "text": "A £500 annual allowance is provided for home office equipment. Purchases must be approved via the procurement portal. Receipts are required."},
    {"id": "expense-meals",     "text": "Meal expenses during business travel are capped at £50 per person per day. Alcohol is not reimbursable. Receipts required for claims over £10."},
    {"id": "expense-travel",    "text": "Economy class is required for flights under 6 hours. Business class may be approved for longer flights. Hotel accommodation is capped at £200 in London."},
    {"id": "salary-review",     "text": "Salary reviews occur every April. Increases are based on performance ratings from the annual review cycle. The typical range is 2-8% for strong performers."},
    {"id": "bonus-policy",      "text": "Discretionary bonuses are paid in December based on company and individual performance. Target bonus is 10% of annual salary for most roles."},
    {"id": "health-benefits",   "text": "All employees receive private health insurance. Dental and vision plans are optional add-ons. Benefits start from the first day of employment."},
    {"id": "pension",           "text": "The company contributes 6% of salary to the pension scheme. Employee contributions start at 3%. The combined 9% minimum meets auto-enrolment requirements."},
    {"id": "training-budget",   "text": "Each employee has a £1,500 annual learning budget for courses, books, and conferences. Manager approval is required for single items over £200."},
]

# Relevance labels: for each query, which document IDs are relevant?
QUERIES = [
    {
        "id": "q1",
        "query": "How much annual leave do I get?",
        "relevant_ids": ["annual-leave"],
    },
    {
        "id": "q2",
        "query": "What leave is available for new parents?",
        "relevant_ids": ["parental-leave", "annual-leave"],
    },
    {
        "id": "q3",
        "query": "Can I claim hotel and meal costs on a business trip?",
        "relevant_ids": ["expense-travel", "expense-meals"],
    },
    {
        "id": "q4",
        "query": "What are the remote work rules for working abroad?",
        "relevant_ids": ["remote-work"],
    },
    {
        "id": "q5",
        "query": "How does the company contribute to my pension?",
        "relevant_ids": ["pension"],
    },
    {
        "id": "q6",
        "query": "What benefits do I get on my first day?",
        "relevant_ids": ["health-benefits"],
    },
    {
        "id": "q7",
        "query": "When do salary increases happen and how much can I expect?",
        "relevant_ids": ["salary-review"],
    },
    {
        "id": "q8",
        "query": "What support is available for professional development?",
        "relevant_ids": ["training-budget"],
    },
]

print(f"Corpus: {len(CORPUS)} documents | Queries: {len(QUERIES)}")

In [ ]:
# Build ChromaDB index
chroma = chromadb.Client()
col = chroma.get_or_create_collection("hr_policy", metadata={"hnsw:space": "cosine"}) 

texts = [d["text"] for d in CORPUS]
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=texts,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)
col.add(
    ids=[d["id"] for d in CORPUS],
    documents=texts,
    embeddings=[e.values for e in result.embeddings],
)


def retrieve(query: str, top_k: int = 5) -> list[str]:
    """Return list of retrieved document IDs in rank order."""
    q_vec = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values
    results = col.query(query_embeddings=[q_vec], n_results=top_k, include=["distances"])
    return results["ids"][0]


print(f"✅ Indexed {col.count()} documents")

---

## 2. Precision@K and Recall@K

In [ ]:
def precision_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    """
    Precision@K = (relevant docs in top-K) / K
    
    Answers: Of the K docs we returned, how many were relevant?
    """
    top_k = retrieved[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant)
    return hits / k


def recall_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    """
    Recall@K = (relevant docs in top-K) / total relevant docs
    
    Answers: Of all relevant docs, how many did we find in the top K?
    """
    if not relevant:
        return 1.0
    top_k = retrieved[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant)
    return hits / len(relevant)


# Evaluate across all queries at K=1, K=3, K=5
print("PRECISION@K AND RECALL@K\n")
print(f"{'Query':<8}  {'P@1':>5} {'P@3':>5} {'P@5':>5}  {'R@1':>5} {'R@3':>5} {'R@5':>5}")
print("-" * 50)

metrics = []
for q in QUERIES:
    retrieved = retrieve(q["query"], top_k=5)
    relevant = q["relevant_ids"]

    p1 = precision_at_k(retrieved, relevant, 1)
    p3 = precision_at_k(retrieved, relevant, 3)
    p5 = precision_at_k(retrieved, relevant, 5)
    r1 = recall_at_k(retrieved, relevant, 1)
    r3 = recall_at_k(retrieved, relevant, 3)
    r5 = recall_at_k(retrieved, relevant, 5)

    metrics.append({"p1": p1, "p3": p3, "p5": p5, "r1": r1, "r3": r3, "r5": r5})
    print(f"  {q['id']}     {p1:.2f}  {p3:.2f}  {p5:.2f}   {r1:.2f}  {r3:.2f}  {r5:.2f}")

# Averages
print("-" * 50)
print(f"  AVG      {mean(m['p1'] for m in metrics):.2f}  {mean(m['p3'] for m in metrics):.2f}  {mean(m['p5'] for m in metrics):.2f}   {mean(m['r1'] for m in metrics):.2f}  {mean(m['r3'] for m in metrics):.2f}  {mean(m['r5'] for m in metrics):.2f}")

print("""
Interpreting trade-offs:
  High P@1, low R@5 → Good at returning ONE right answer but misses others
  Low P@3, high R@3 → Returns relevant docs but buries them in noise
  
  For RAG: optimise R@K (find the right docs) > P@K (noise hurts less if you compress)
""")

---

## 3. Mean Reciprocal Rank (MRR)

In [ ]:
def reciprocal_rank(retrieved: list[str], relevant: list[str]) -> float:
    """
    Reciprocal Rank = 1 / rank_of_first_relevant_result
    
    Answers: How high is the first relevant document in the ranking?
    RR=1.0 → first result is relevant
    RR=0.5 → second result is relevant  
    RR=0.33 → third result is relevant
    RR=0.0 → no relevant result found
    """
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def mean_reciprocal_rank(query_results: list[tuple[list[str], list[str]]]) -> float:
    """MRR = average reciprocal rank across all queries."""
    rrs = [reciprocal_rank(retrieved, relevant) for retrieved, relevant in query_results]
    return mean(rrs)


print("MEAN RECIPROCAL RANK (MRR)\n")
print(f"{'Query':<8}  {'RR':>6}  First relevant result")
print("-" * 60)

all_results = []
for q in QUERIES:
    retrieved = retrieve(q["query"], top_k=10)
    relevant = q["relevant_ids"]
    rr = reciprocal_rank(retrieved, relevant)
    all_results.append((retrieved, relevant))

    # Find position of first relevant doc
    first_pos = next(
        (i+1 for i, doc_id in enumerate(retrieved) if doc_id in relevant), None
    )
    first_doc = next(
        (doc_id for doc_id in retrieved if doc_id in relevant), "not found"
    )
    print(f"  {q['id']}      {rr:.3f}  rank #{first_pos}: {first_doc}")

mrr = mean_reciprocal_rank(all_results)
print("-" * 60)
print(f"  MRR: {mrr:.3f}")
print(f"""
MRR interpretation:
  1.00 = First result is always relevant (perfect)
  0.50 = First relevant result is at rank 2 on average
  0.33 = First relevant result is at rank 3 on average
  < 0.25 = Users need to scroll significantly to find relevant content

Current MRR: {mrr:.3f} → first relevant result is at avg rank {1/mrr:.1f}
""")

---

## 4. Normalised Discounted Cumulative Gain (NDCG)

When documents have **graded relevance** (e.g., "very relevant", "somewhat relevant", "not relevant")  
rather than binary, NDCG is the right metric.

In [ ]:
def dcg_at_k(relevance_scores: list[int], k: int) -> float:
    """
    Discounted Cumulative Gain at K.
    Rewards placing highly-relevant documents early in the ranking.
    relevance_scores[i] = relevance score of the i-th retrieved document (0, 1, or 2)
    """
    dcg = 0.0
    for i, rel in enumerate(relevance_scores[:k]):
        # Standard formula: rel / log2(rank + 1)
        dcg += rel / math.log2(i + 2)  # i+2 because log2(1)=0
    return dcg


def ndcg_at_k(retrieved: list[str], relevance_map: dict[str, int], k: int) -> float:
    """
    NDCG@K = DCG / ideal DCG
    relevance_map: {doc_id: relevance_score}
    """
    # Scores for retrieved docs
    retrieved_scores = [relevance_map.get(doc_id, 0) for doc_id in retrieved[:k]]

    # Ideal ordering: all relevant docs first, sorted by score
    ideal_scores = sorted(relevance_map.values(), reverse=True)

    dcg = dcg_at_k(retrieved_scores, k)
    idcg = dcg_at_k(ideal_scores, k)

    return dcg / idcg if idcg > 0 else 0.0


# Graded relevance example: parental leave query
# parental-leave = 2 (directly about parental leave)
# annual-leave = 1 (somewhat relevant — may be used alongside parental leave)
# others = 0

GRADED_QUERIES = [
    {
        "id": "q2-graded",
        "query": "What leave is available for new parents?",
        "relevance_map": {"parental-leave": 2, "annual-leave": 1},
    },
    {
        "id": "q3-graded",
        "query": "Can I claim hotel and meal costs on a business trip?",
        "relevance_map": {"expense-travel": 2, "expense-meals": 2},
    },
]

print("NDCG@5 WITH GRADED RELEVANCE\n")
print(f"{'Query':<14}  {'NDCG@1':>8} {'NDCG@3':>8} {'NDCG@5':>8}")
print("-" * 45)

for q in GRADED_QUERIES:
    retrieved = retrieve(q["query"], top_k=5)
    n1 = ndcg_at_k(retrieved, q["relevance_map"], 1)
    n3 = ndcg_at_k(retrieved, q["relevance_map"], 3)
    n5 = ndcg_at_k(retrieved, q["relevance_map"], 5)
    print(f"  {q['id']:<12}    {n1:.4f}   {n3:.4f}   {n5:.4f}")
    print(f"    Retrieved: {retrieved}")

print("""
NDCG=1.0 = Perfect — most relevant docs ranked first
NDCG<0.5 = Significant ordering or coverage problems
""")

---

## 5. Context Relevance — LLM-Judged Retrieval Quality

The metrics above assume you have relevance labels.  
**Context relevance** uses an LLM to judge whether retrieved chunks are useful — no labels needed.

In [ ]:
class ContextRelevance(BaseModel):
    is_relevant: bool
    relevance_score: int  # 1–3: 1=not relevant, 2=partially, 3=directly relevant
    reasoning: str


def score_context_relevance(query: str, context_chunk: str) -> ContextRelevance:
    """Ask LLM whether a retrieved chunk would help answer the query."""
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Does the following context chunk contain information that would help answer the query?

Query: {query}

Context chunk: {context_chunk}

Score:
3 = Directly relevant — chunk contains the answer or key information needed
2 = Partially relevant — chunk has some related information but not the core answer
1 = Not relevant — chunk does not help answer the query""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ContextRelevance,
            temperature=0.0,
        ),
    )
    return ContextRelevance.model_validate_json(response.text)


def evaluate_retrieval_context_relevance(query: str, top_k: int = 3) -> dict:
    """Retrieve top-K and score each chunk's relevance."""
    q_vec = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values
    results = col.query(
        query_embeddings=[q_vec],
        n_results=top_k,
        include=["documents"],
    )

    chunk_scores = []
    for doc_id, doc_text in zip(results["ids"][0], results["documents"][0]):
        score = score_context_relevance(query, doc_text)
        chunk_scores.append({"id": doc_id, "score": score.relevance_score, "reason": score.reasoning})

    avg_relevance = mean(c["score"] for c in chunk_scores) / 3  # normalise to 0-1
    return {"query": query, "chunks": chunk_scores, "avg_relevance": avg_relevance}


# Evaluate context relevance for 3 queries
test_queries = [
    "How much annual leave do I get?",
    "What support is available for professional development?",
    "Can I claim hotel and meal costs on a business trip?",
]

print("CONTEXT RELEVANCE (LLM-judged)\n")
for q_text in test_queries:
    result = evaluate_retrieval_context_relevance(q_text, top_k=3)
    print(f"Query: '{q_text}'")
    print(f"  Avg context relevance: {result['avg_relevance']:.2f}")
    for c in result["chunks"]:
        print(f"  [{c['score']}/3] {c['id']}: {c['reason'][:70]}")
    print()

---

## 6. Complete Retrieval Evaluation Report

In [ ]:
def retrieval_evaluation_report(queries: list[dict], top_k: int = 3) -> dict:
    """
    Run a full retrieval evaluation and return a report dict.
    Computes: Precision@K, Recall@K, MRR across all queries.
    """
    p_at_k_scores = []
    r_at_k_scores = []
    rr_scores = []
    per_query = []

    for q in queries:
        retrieved = retrieve(q["query"], top_k=top_k)
        relevant = q["relevant_ids"]

        p = precision_at_k(retrieved, relevant, top_k)
        r = recall_at_k(retrieved, relevant, top_k)
        rr = reciprocal_rank(retrieved, relevant)

        p_at_k_scores.append(p)
        r_at_k_scores.append(r)
        rr_scores.append(rr)

        per_query.append({
            "id": q["id"],
            "query": q["query"],
            "precision": p,
            "recall": r,
            "rr": rr,
            "retrieved": retrieved,
            "relevant": relevant,
        })

    return {
        "k": top_k,
        "num_queries": len(queries),
        "mean_precision": mean(p_at_k_scores),
        "mean_recall": mean(r_at_k_scores),
        "mrr": mean(rr_scores),
        "per_query": per_query,
    }


# Run evaluation at K=3
report = retrieval_evaluation_report(QUERIES, top_k=3)

print(f"RETRIEVAL EVALUATION REPORT (K={report['k']})")
print(f"Queries: {report['num_queries']}")
print(f"─" * 60)
print(f"  Mean Precision@{report['k']}: {report['mean_precision']:.3f}")
print(f"  Mean Recall@{report['k']}:    {report['mean_recall']:.3f}")
print(f"  MRR:              {report['mrr']:.3f}")
print(f"─" * 60)
print()

# Identify failures
failures = [q for q in report["per_query"] if q["recall"] == 0.0]
if failures:
    print(f"⚠️  Complete retrieval failures ({len(failures)} queries — relevant doc not in top-{report['k']}):")
    for f in failures:
        print(f"  [{f['id']}] '{f['query']}'")
        print(f"    Expected: {f['relevant']}")
        print(f"    Got:      {f['retrieved'][:3]}")

---

## 🏋️ Exercise 1: Compare Retrieval Strategies

Measure whether retrieval quality improves when you change the `top_k` value.

In [ ]:
# Exercise 1: K sensitivity analysis
# TODO:
# 1. Run retrieval_evaluation_report() at K=1, 3, 5, 10
# 2. Plot (or print) the trade-off curve:
#    - As K increases, Recall should go UP
#    - As K increases, Precision should go DOWN
#    - MRR should stay roughly stable (it's position-based)
# 3. Find the K value with the best F1 score (harmonic mean of Precision and Recall)
# 4. Write a recommendation: what K would you use in production and why?

k_values = [1, 3, 5, 10]

print(f"{'K':>4}  {'Precision':>10} {'Recall':>8} {'MRR':>7} {'F1':>7}")
print("-" * 45)
# TODO: Run evaluations and print results

# RECOMMENDATION:
# TODO: Write 2-3 sentences recommending a K value for this corpus

---

## 🏋️ Exercise 2: Diagnose Retrieval Failures

For each query where the relevant document wasn't retrieved, figure out *why*.

In [ ]:
# Exercise 2: Root cause analysis
# Use the report from above to identify any queries where Recall@3 == 0
# (relevant doc was NOT found in the top 3 results)
#
# For each failure:
# 1. Print the query, what was retrieved, and what was expected
# 2. Look at the text of the expected document vs the retrieved documents
# 3. Hypothesise why the retrieval failed:
#    - Is it a keyword mismatch? (query uses different vocabulary)
#    - Is it a topic confusion? (retrieved doc is thematically similar but wrong)
#    - Is the expected document text poorly written for search?
# 4. Propose a fix for each failure — could be:
#    - Rephrase the document to match likely query vocabulary
#    - Add a query rewriting step (from Day 8)
#    - Add keyword-based (BM25) retrieval as a complement

# Identify failures from your K=3 report
failures = [q for q in report["per_query"] if q["recall"] < 1.0]

print(f"Queries with Recall@3 < 1.0: {len(failures)}\n")
for f in failures:
    print(f"  Query: {f['query']}")
    print(f"  Expected: {f['relevant']}")
    print(f"  Retrieved: {f['retrieved']}")
    # TODO: Look up the text of expected vs retrieved docs
    # TODO: Hypothesise why it failed
    # TODO: Propose a fix
    print()

---

## Key Takeaways

| Metric | Formula | What It Measures |
|--------|---------|------------------|
| Precision@K | (relevant in top-K) / K | Signal-to-noise ratio of retrieved results |
| Recall@K | (relevant in top-K) / (total relevant) | Coverage — did we find all relevant docs? |
| MRR | avg(1 / rank of first relevant) | How quickly the first useful result appears |
| NDCG@K | normalised discounted gain | Graded relevance with position discount |
| Context Relevance | LLM-judged | Is retrieved content actually useful? (no labels needed) |

**When to use each:**
- Early-stage development → Precision@3 + Recall@3 (fast, interpretable)
- Search UX → MRR (users care about the first useful result)
- Multiple-relevant-docs scenarios → NDCG
- No labels available → LLM-judged context relevance

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [Information Retrieval Metrics](https://en.wikipedia.org/wiki/Evaluation_measures_(information_retrieval)) | Wiki | Full taxonomy of IR metrics |
| [RAGAS Context Precision](https://docs.ragas.io/en/stable/concepts/metrics/context_precision.html) | Docs | RAGAS framework's context evaluation |
| [BEIR Benchmark](https://arxiv.org/abs/2104.08663) | Paper | 18-dataset retrieval evaluation benchmark |

---

**Next up:** [04_Building_an_Evaluation_Pipeline.ipynb](./04_Building_an_Evaluation_Pipeline.ipynb)